# Nominal one-year cash-flow growth — Shiller vs De La O–Myers

Build nominal one-year **dividend** and **earnings** growth from Shiller's long
dataset (`ie_data.xls`, 1871–) and validate the *realized* growth against the
realized columns of the De La O & Myers replication files.

All series are **nominal**. Everything is quarterly and merged on
`(Year, Quarter)` — no month-mapping conventions.

**Outputs** (in `~/Shiller_div_earnings`):

- `shiller_growth_quarterly.csv` — quarterly `D`, `E`, one-year log growth (trailing & forward), 1871–
- `comparison_dividends.csv`, `comparison_earnings.csv` — quarterly DLM vs Shiller
- `fig_timeseries.pdf`, `fig_scatter.pdf`

In [ ]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt

%matplotlib inline
warnings.simplefilter("ignore")

F = "/home/rpa9/Shiller_div_earnings"


## 1. Shiller series, collapsed to quarterly

`ie_data.xls` has seven header rows; monthly data start at row 8 (1871.01).
Shiller's `D` and `E` are **annual** dividend / earnings per index share, reported
monthly by interpolation — so they are taken directly, never summed across months.
We keep the quarter-end month (Mar/Jun/Sep/Dec) as the quarterly observation.

In [ ]:
m = pd.read_excel(f"{F}/ie_data.xls", sheet_name="Data", header=None, skiprows=8)
m = m[m[0].notna()].iloc[:, :5]
m.columns = ["raw_date", "P", "D", "E", "CPI"]
for c in ["P", "D", "E", "CPI"]:
    m[c] = pd.to_numeric(m[c], errors="coerce")
m["date"] = pd.date_range("1871-01-01", periods=len(m), freq="MS")

# one row per quarter: keep the quarter-end month
shq = m[m["date"].dt.month.isin([3, 6, 9, 12])].copy()
shq["Year"] = shq["date"].dt.year
shq["Quarter"] = shq["date"].dt.quarter
shq = shq[["Year", "Quarter", "date", "D", "E"]].reset_index(drop=True)

print(f"Shiller quarterly: {len(shq)} rows | "
      f"{shq.Year.iloc[0]}Q{shq.Quarter.iloc[0]} .. {shq.Year.iloc[-1]}Q{shq.Quarter.iloc[-1]}")
shq.head()


## 2. One-year log growth

One-year growth is the year-over-year (4-quarter) log change of the annual
series — not an adjacent-quarter difference.

- `*_trail` — trailing: growth realized over the year *ending* at quarter *t*
- `*_fwd` — forward: growth realized over the year *starting* at quarter *t*

DLM's "realized next year" column is a **forward** object, so `*_fwd` is the
series that should line up with it. By construction `*_fwd(t)` = `*_trail(t+4)`.

In [ ]:
# one-year (4-quarter) log growth of the annual series
shq["g_div_trail"]  = np.log(shq.D) - np.log(shq.D.shift(4))
shq["g_div_fwd"]    = np.log(shq.D.shift(-4)) - np.log(shq.D)
shq["g_earn_trail"] = np.log(shq.E) - np.log(shq.E.shift(4))
shq["g_earn_fwd"]   = np.log(shq.E.shift(-4)) - np.log(shq.E)

shq.to_csv(f"{F}/shiller_growth_quarterly.csv", index=False)
shq[["Year", "Quarter", "D", "E",
     "g_div_trail", "g_div_fwd", "g_earn_trail", "g_earn_fwd"]].tail()


## 3. De La O–Myers replication files

Both files carry the *expected* and *realized* one-year growth. The earnings file
has a two-row header and three denominator blocks; we keep the standard block
(denominator = current earnings $e_t$), which is its first four columns.

In [ ]:
dv = pd.read_excel(f"{F}/Dividend_growth_expectations.xlsx")
dv.columns = ["Year", "Quarter", "dlm_exp", "dlm_real", "dlm_pd"]

en = pd.read_excel(f"{F}/Earnings_growth_expectations.xlsx",
                   header=None, skiprows=2).iloc[:, :4]
en.columns = ["Year", "Quarter", "dlm_exp", "dlm_real"]

for d in (dv, en):
    d["Year"] = d["Year"].astype(int)
    d["Quarter"] = d["Quarter"].astype(int)

print("dividends:", dv.shape, " earnings:", en.shape)
dv.head()


## 4. Merge on `(Year, Quarter)`

A direct exact-key merge — no dates assigned to DLM, no month convention.
Shiller forward growth pairs with DLM realized; Shiller trailing is carried
along for the timing check in section 5.

In [ ]:
md = dv.merge(shq[["Year", "Quarter", "date", "g_div_fwd", "g_div_trail"]],
              on=["Year", "Quarter"], how="left").rename(
              columns={"g_div_fwd": "shiller_fwd", "g_div_trail": "shiller_trail"})
me = en.merge(shq[["Year", "Quarter", "date", "g_earn_fwd", "g_earn_trail"]],
              on=["Year", "Quarter"], how="left").rename(
              columns={"g_earn_fwd": "shiller_fwd", "g_earn_trail": "shiller_trail"})

md.to_csv(f"{F}/comparison_dividends.csv", index=False)
me.to_csv(f"{F}/comparison_earnings.csv", index=False)
md.head()


## 5. Timing check and fit

If DLM's realized column is the *next-year* (forward) window, then
`corr(DLM realized, Shiller forward)` should clearly beat
`corr(DLM realized, Shiller trailing)`. Then report the fit of the forward
series: full-sample correlation, a crisis-robust correlation, and mean bias.

In [ ]:
def assess(m, label):
    s = m.dropna(subset=["dlm_real", "shiller_fwd", "shiller_trail"])
    c_fwd   = s["dlm_real"].corr(s["shiller_fwd"])
    c_trail = s["dlm_real"].corr(s["shiller_trail"])
    diff = s["shiller_fwd"] - s["dlm_real"]
    core = s[s["dlm_real"].abs() < 0.5]
    print(f"[{label}]  n={len(s)}  "
          f"{s.Year.iloc[0]}Q{s.Quarter.iloc[0]}..{s.Year.iloc[-1]}Q{s.Quarter.iloc[-1]}")
    print(f"  corr(DLM realized, Shiller FORWARD)   = {c_fwd:7.4f}   <- aligned window")
    print(f"  corr(DLM realized, Shiller TRAILING)  = {c_trail:7.4f}")
    if len(core) > 8:
        print(f"  corr forward, excl |g| > 0.50         = "
              f"{core['dlm_real'].corr(core['shiller_fwd']):7.4f}   (n={len(core)})")
    print(f"  mean  DLM = {s['dlm_real'].mean():+.4f}    Shiller = {s['shiller_fwd'].mean():+.4f}")
    print(f"  mean diff (Sh-DLM) = {diff.mean():+.4f}    RMSE = {np.sqrt((diff ** 2).mean()):.4f}\n")

assess(md, "DIVIDENDS")
assess(me, "EARNINGS")


## 6. Figures

### Time series

In [ ]:
plt.rcParams.update({"font.family": "serif", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 120})
C_DLM, C_SH, C_EXP = "#1f3b73", "#c0392b", "#9aa7bd"

fig, ax = plt.subplots(2, 1, figsize=(10, 8))
for a, m, ttl in [(ax[0], md, "Dividend growth"), (ax[1], me, "Earnings growth")]:
    s = m.dropna(subset=["dlm_real"])
    a.plot(s.date, s.dlm_exp,  color=C_EXP, lw=1.0, label="DLM expected (1-yr)")
    a.plot(s.date, s.dlm_real, color=C_DLM, lw=1.8, label="DLM realized (next-yr)")
    a.plot(m.date, m.shiller_fwd, color=C_SH, lw=1.5, ls="--",
           label="Shiller realized (next-yr)")
    fit = m.dropna(subset=["dlm_real", "shiller_fwd"])
    c = fit["dlm_real"].corr(fit["shiller_fwd"])
    a.axhline(0, color="k", lw=0.6)
    a.set_title(f"{ttl}  —  realized: corr(DLM, Shiller) = {c:.3f}", fontsize=10.5)
    a.set_ylabel("one-year log growth")
    a.legend(frameon=False, fontsize=8.5, ncol=3, loc="lower center")
ax[1].set_xlabel("date (start of realization window)")
fig.suptitle("Nominal one-year cash-flow growth: Shiller long series vs De La O–Myers",
             fontsize=12, y=0.995)
fig.tight_layout()
fig.savefig(f"{F}/fig_timeseries.pdf", bbox_inches="tight")
plt.show()


### Scatter

In [ ]:
fig2, ax2 = plt.subplots(1, 2, figsize=(11, 5.2))
for a, m, ttl in [(ax2[0], md, "Dividends"), (ax2[1], me, "Earnings")]:
    s = m.dropna(subset=["dlm_real", "shiller_fwd"])
    a.scatter(s.dlm_real, s.shiller_fwd, s=22, color=C_DLM,
              alpha=0.7, edgecolor="w", lw=0.4)
    lo = min(s.dlm_real.min(), s.shiller_fwd.min())
    hi = max(s.dlm_real.max(), s.shiller_fwd.max())
    a.plot([lo, hi], [lo, hi], color=C_SH, lw=1.2, ls="--", label="45°")
    a.set_xlabel("DLM realized one-year growth")
    a.set_ylabel("Shiller realized one-year growth")
    a.set_title(f"{ttl}  (corr = {s['dlm_real'].corr(s['shiller_fwd']):.3f}, n={len(s)})")
    a.legend(frameon=False, fontsize=9)
fig2.suptitle("Realized one-year growth: Shiller vs De La O–Myers", fontsize=12)
fig2.tight_layout()
fig2.savefig(f"{F}/fig_scatter.pdf", bbox_inches="tight")
plt.show()
